# LeetCode 501 — Find Mode in Binary Search Tree bằng Python

**Link đề:** https://leetcode.com/problems/find-mode-in-binary-search-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Binary Search Tree, Binary Tree

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findMode(self, root: TreeNode | None) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu **mode** (giá trị xuất hiện nhiều nhất) và **BST cho phép trùng** (`<=` bên trái, `>=` bên phải).
2. Tự dựng `TreeNode`, hàm dựng cây từ list level-order và hàm in cây để notebook chạy độc lập.
3. Cách 1: duyệt bất kỳ + `Counter` (không dùng tính chất BST).
4. Insight: **duyệt in-order của BST cho dãy không giảm** → các giá trị bằng nhau nằm **liền nhau**.
5. Cách 2–3: in-order + đếm "chuỗi bằng nhau" với `prev`, `count`, `max_count` — không cần hash map. Cách 4 (nâng cao): **Morris traversal** bộ nhớ O(1) cho follow up.

## 1. Đề bài

Cho `root` của một **cây nhị phân tìm kiếm (BST) có thể chứa giá trị trùng**, trả về **tất cả mode** — giá trị xuất hiện **nhiều lần nhất**. Nếu có nhiều mode, trả theo **thứ tự bất kỳ**.

BST trong bài được định nghĩa:
- Cây con trái chứa các key **≤** key của node.
- Cây con phải chứa các key **≥** key của node.
- Cả hai cây con cũng là BST.

| root (level order) | Output |
|---|---|
| `[1,null,2,2]` | `[2]` |
| `[0]` | `[0]` |

```text
Ví dụ 1:   1
            \
             2
            /
           2
```
Giá trị 2 xuất hiện 2 lần, giá trị 1 xuất hiện 1 lần → mode là `[2]`.

**Constraints:** số node trong `[1, 10^4]`; `-10^5 <= Node.val <= 10^5`.

**Follow up:** làm được **không dùng thêm bộ nhớ** (không tính stack đệ quy) không?

## 2. Nền tảng: `TreeNode`, dựng cây và in cây

LeetCode cho cây dưới dạng list **level order** (theo tầng, từ trái sang phải), `null` (Python: `None`) đánh dấu vị trí không có node.
Ta tự viết:
- `TreeNode`: một node có `val`, `left`, `right`.
- `build_tree(values)`: dựng cây từ list level order dùng **hàng đợi** (`collections.deque`).
- `show_tree(root)`: in cây nằm ngang (gốc bên trái, cây con phải ở trên) để dễ nhìn.

In [1]:
from collections import deque, Counter

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    # Dựng cây từ list level-order kiểu LeetCode (None = không có node).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def show_tree(node, depth=0):
    # In cây nằm ngang: cây con phải ở trên, cây con trái ở dưới.
    if node is None:
        return
    show_tree(node.right, depth + 1)
    print("    " * depth + str(node.val))
    show_tree(node.left, depth + 1)

root = build_tree([1, None, 2, 2])
show_tree(root)

    2
        2
1


### 2.1. `deque` — hàng đợi hai đầu

`deque` (đọc là "deck") trong `collections` hỗ trợ `append` (thêm cuối) và `popleft` (lấy đầu) đều O(1).
Dùng list với `pop(0)` cũng được nhưng tốn O(n) mỗi lần vì phải dời các phần tử.

In [2]:
q = deque([1, 2])
q.append(3)
print(q.popleft(), q)   # 1 deque([2, 3])

1 deque([2, 3])


### 2.2. Đệ quy duyệt cây — 3 thứ tự

- **Pre-order**: node → trái → phải.
- **In-order**: trái → node → phải. ⭐ Với BST, in-order cho dãy **không giảm**.
- **Post-order**: trái → phải → node.

Hàm đệ quy cần **điểm dừng** (base case): `if node is None: return`.

In [3]:
def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def preorder(node, out):
    if node is None:
        return
    out.append(node.val)
    preorder(node.left, out)
    preorder(node.right, out)

bst = build_tree([5, 3, 7, 3, 4, 6, 7])
show_tree(bst)
a, p = [], []
inorder(bst, a); preorder(bst, p)
print("in-order :", a)
print("pre-order:", p)

        7
    7
        6
5
        4
    3
        3
in-order : [3, 3, 4, 5, 6, 7, 7]
pre-order: [5, 3, 3, 4, 7, 6, 7]


Thấy không: in-order cho `[3, 3, 4, 5, 6, 7, 7]` — đã sắp xếp, và hai số `3` đứng **liền nhau**, hai số `7` đứng liền nhau.

### 2.3. `Counter` — đếm tần suất

`Counter(iterable)` là dict đặc biệt: key = phần tử, value = số lần xuất hiện. `max(c.values())` lấy tần suất lớn nhất.

In [4]:
c = Counter([3, 3, 4, 5, 6, 7, 7])
print(c)
best = max(c.values())
print("tần suất lớn nhất:", best)
print("các mode:", [v for v, f in c.items() if f == best])

Counter({3: 2, 7: 2, 4: 1, 5: 1, 6: 1})
tần suất lớn nhất: 2
các mode: [3, 7]


## 3. Cách 1 — Trực giác: duyệt hết + đếm bằng `Counter`

Không cần biết gì về BST: thu mọi giá trị, đếm, lấy các giá trị có tần suất cực đại.

In [5]:
def find_mode_counter(root):
    values = []
    preorder(root, values)          # thứ tự nào cũng được
    freq = Counter(values)
    best = max(freq.values())
    return [v for v, f in freq.items() if f == best]

print(find_mode_counter(build_tree([1, None, 2, 2])))   # [2]
print(find_mode_counter(build_tree([0])))               # [0]
print(find_mode_counter(bst))                           # [3, 7]

[2]
[0]
[3, 7]


### Nút thắt

- Cách 1 tốn `O(n)` bộ nhớ: list `values` + dict `Counter` với tối đa `n` key.
- Ta **chưa tận dụng** việc cây là BST. Follow up muốn bộ nhớ phụ `O(1)` (không tính stack đệ quy).

Câu hỏi: nếu dãy giá trị đã **sắp xếp**, ta có cần dict để đếm không?

## 4. Insight — dãy sắp xếp thì các giá trị bằng nhau liền kề

Với dãy sắp xếp `[3, 3, 4, 5, 6, 7, 7]`, đếm tần suất = đếm độ dài **các đoạn bằng nhau liên tiếp** (run). Chỉ cần nhớ:
- `prev`: giá trị vừa thấy,
- `count`: độ dài đoạn hiện tại,
- `max_count`: độ dài đoạn dài nhất đã thấy,
- `modes`: các giá trị có đoạn dài bằng `max_count`.

Quy tắc khi gặp giá trị `v`:
1. `count = count + 1` nếu `v == prev`, ngược lại `count = 1`.
2. Nếu `count > max_count`: kỷ lục mới → `max_count = count`, `modes = [v]`.
3. Nếu `count == max_count`: hoà kỷ lục → `modes.append(v)`.

In [6]:
def modes_of_sorted(arr):
    prev, count, max_count, modes = None, 0, 0, []
    for v in arr:
        count = count + 1 if v == prev else 1
        if count > max_count:
            max_count, modes = count, [v]
        elif count == max_count:
            modes.append(v)
        prev = v
    return modes

print(modes_of_sorted([3, 3, 4, 5, 6, 7, 7]))   # [3, 7]
print(modes_of_sorted([1, 2, 2]))               # [2]
print(modes_of_sorted([1, 1, 1, 2, 2, 2, 2]))   # [2]

[3, 7]
[2]
[2]


Lưu ý thứ tự cập nhật: khi đoạn `7, 7` đạt `count = 2` bằng kỷ lục → thêm 7. Nếu sau đó còn một số `7` nữa (`count = 3 > 2`) → reset `modes = [7]`. Mọi trường hợp đều được xử lý đúng lúc.

## 5. Cách 2 — In-order ra list rồi đếm đoạn

Ghép insight với tính chất BST: `inorder` cho dãy sắp xếp, rồi gọi `modes_of_sorted`.

In [7]:
def find_mode_inorder_list(root):
    arr = []
    inorder(root, arr)
    return modes_of_sorted(arr)

print(find_mode_inorder_list(build_tree([1, None, 2, 2])))
print(find_mode_inorder_list(bst))

[2]
[3, 7]


Vẫn tốn `O(n)` cho `arr`. Bước tiếp theo: **xử lý ngay khi thăm node**, không lưu list.

## 6. Cách 3 — Lời giải chính: in-order + đếm "tại chỗ"

Các đoạn:
1. Trạng thái dùng chung: `prev`, `count`, `max_count`, `modes`. Trong hàm lồng nhau, muốn **gán lại** biến của hàm ngoài phải khai báo `nonlocal`.
2. `visit(v)`: đúng 3 quy tắc ở mục 4.
3. `dfs(node)`: in-order — `dfs(left)`, `visit(node.val)`, `dfs(right)`.

In [8]:
# Thí nghiệm nhỏ về nonlocal
def outer():
    count = 0
    def inc():
        nonlocal count      # không có dòng này -> UnboundLocalError
        count += 1
    inc(); inc()
    return count

print(outer())   # 2

2


In [9]:
def find_mode_inorder_inplace(root):
    prev, count, max_count, modes = None, 0, 0, []

    def visit(v):
        nonlocal prev, count, max_count, modes
        count = count + 1 if v == prev else 1
        if count > max_count:
            max_count = count
            modes = [v]
        elif count == max_count:
            modes.append(v)
        prev = v

    def dfs(node):
        if node is None:
            return
        dfs(node.left)
        visit(node.val)
        dfs(node.right)

    dfs(root)
    return modes

print(find_mode_inorder_inplace(build_tree([1, None, 2, 2])))   # [2]
print(find_mode_inorder_inplace(build_tree([0])))               # [0]
print(find_mode_inorder_inplace(bst))                           # [3, 7]

[2]
[0]
[3, 7]


## 7. TRACE từng bước

Cây `[5, 3, 7, 3, 4, 6, 7]`, in-order là `3, 3, 4, 5, 6, 7, 7`:

| bước | v | prev trước | count | max_count | quyết định | modes |
|---:|---:|---|---:|---:|---|---|
| 1 | 3 | None | 1 | 1 | kỷ lục mới | `[3]` |
| 2 | 3 | 3 | 2 | 2 | kỷ lục mới | `[3]` |
| 3 | 4 | 3 | 1 | 2 | thấp hơn | `[3]` |
| 4 | 5 | 4 | 1 | 2 | thấp hơn | `[3]` |
| 5 | 6 | 5 | 1 | 2 | thấp hơn | `[3]` |
| 6 | 7 | 6 | 1 | 2 | thấp hơn | `[3]` |
| 7 | 7 | 7 | 2 | 2 | hoà kỷ lục | `[3, 7]` |

In [10]:
def trace_find_mode(root):
    state = {"prev": None, "count": 0, "max": 0, "modes": [], "step": 0}

    def visit(v):
        s = state
        s["step"] += 1
        prev_before = s["prev"]
        s["count"] = s["count"] + 1 if v == s["prev"] else 1
        if s["count"] > s["max"]:
            s["max"] = s["count"]; s["modes"] = [v]; why = "kỷ lục mới"
        elif s["count"] == s["max"]:
            s["modes"].append(v); why = "hoà kỷ lục"
        else:
            why = "thấp hơn"
        s["prev"] = v
        print(f"bước {s['step']}: v={v:>3} prev={str(prev_before):>4} count={s['count']} "
              f"max={s['max']} -> {why:10s} modes={s['modes']}")

    def dfs(node):
        if node is None:
            return
        dfs(node.left)
        visit(node.val)
        dfs(node.right)

    dfs(root)
    return state["modes"]

trace_find_mode(build_tree([5, 3, 7, 3, 4, 6, 7]))

bước 1: v=  3 prev=None count=1 max=1 -> kỷ lục mới modes=[3]
bước 2: v=  3 prev=   3 count=2 max=2 -> kỷ lục mới modes=[3]
bước 3: v=  4 prev=   3 count=1 max=2 -> thấp hơn   modes=[3]
bước 4: v=  5 prev=   4 count=1 max=2 -> thấp hơn   modes=[3]
bước 5: v=  6 prev=   5 count=1 max=2 -> thấp hơn   modes=[3]
bước 6: v=  7 prev=   6 count=1 max=2 -> thấp hơn   modes=[3]
bước 7: v=  7 prev=   7 count=2 max=2 -> hoà kỷ lục modes=[3, 7]


[3, 7]

(Ở hàm trace, ta dùng dict `state` thay cho `nonlocal` — một cách khác để hàm con sửa trạng thái chung: sửa **nội dung** dict thì không cần `nonlocal`.)

Trace thêm một cây có **kỷ lục bị phá** sau khi đã hoà, để thấy `modes` bị reset:

In [11]:
t = build_tree([2, 1, 3, 1, 2, 3, 3])
show_tree(t)
trace_find_mode(t)

        3
    3
        3
2
        2
    1
        1
bước 1: v=  1 prev=None count=1 max=1 -> kỷ lục mới modes=[1]
bước 2: v=  1 prev=   1 count=2 max=2 -> kỷ lục mới modes=[1]
bước 3: v=  2 prev=   1 count=1 max=2 -> thấp hơn   modes=[1]
bước 4: v=  2 prev=   2 count=2 max=2 -> hoà kỷ lục modes=[1, 2]
bước 5: v=  3 prev=   2 count=1 max=2 -> thấp hơn   modes=[1, 2]
bước 6: v=  3 prev=   3 count=2 max=2 -> hoà kỷ lục modes=[1, 2, 3]
bước 7: v=  3 prev=   3 count=3 max=3 -> kỷ lục mới modes=[3]


[3]

## 8. Cách 4 (nâng cao) — Morris in-order: bộ nhớ phụ O(1) thật sự

Đệ quy tốn stack `O(h)` (h = chiều cao). Morris traversal duyệt in-order **không stack**, bằng cách tạm "nối" node tiền nhiệm (predecessor — node phải nhất của cây con trái) về node hiện tại, rồi gỡ ra.

```text
cur = root
while cur:
    nếu cur không có con trái: thăm cur, cur = cur.right
    ngược lại: pre = node phải nhất của cây con trái (dừng nếu pre.right == cur)
        nếu pre.right is None: pre.right = cur (nối tạm), cur = cur.left
        nếu pre.right == cur : pre.right = None (gỡ), thăm cur, cur = cur.right
```
Cây được **khôi phục nguyên vẹn** sau khi chạy xong.

In [12]:
def find_mode_morris(root):
    prev, count, max_count, modes = None, 0, 0, []

    def visit(v):
        nonlocal prev, count, max_count, modes
        count = count + 1 if v == prev else 1
        if count > max_count:
            max_count, modes = count, [v]
        elif count == max_count:
            modes.append(v)
        prev = v

    cur = root
    while cur:
        if cur.left is None:
            visit(cur.val)
            cur = cur.right
        else:
            pre = cur.left
            while pre.right is not None and pre.right is not cur:
                pre = pre.right
            if pre.right is None:
                pre.right = cur          # nối tạm
                cur = cur.left
            else:
                pre.right = None         # gỡ nối
                visit(cur.val)
                cur = cur.right
    return modes

t = build_tree([5, 3, 7, 3, 4, 6, 7])
print(find_mode_morris(t))
after = []; inorder(t, after)
print("cây sau Morris vẫn nguyên:", after)

[3, 7]
cây sau Morris vẫn nguyên: [3, 3, 4, 5, 6, 7, 7]


## 9. Code nộp LeetCode

### Bản Counter (đơn giản nhất)

In [13]:
class Solution:
    def findMode(self, root: TreeNode | None) -> list[int]:
        freq = Counter()
        stack = [root]
        while stack:                     # duyệt bằng stack, tránh đệ quy sâu
            node = stack.pop()
            if node:
                freq[node.val] += 1
                stack.append(node.left)
                stack.append(node.right)
        best = max(freq.values())
        return [v for v, f in freq.items() if f == best]

print(Solution().findMode(build_tree([1, None, 2, 2])))

[2]


### Bản in-order + đếm đoạn (khuyên dùng, tận dụng BST)

Với tối đa `10^4` node, cây có thể "lệch" thành một đường thẳng cao `10^4` — vượt giới hạn đệ quy mặc định của Python (1000). Vì vậy bản nộp dùng **in-order bằng stack** (lặp), an toàn hơn đệ quy.

In [14]:
class Solution:
    def findMode(self, root: TreeNode | None) -> list[int]:
        prev, count, max_count, modes = None, 0, 0, []
        stack, cur = [], root
        while stack or cur:
            while cur:                    # đi hết về bên trái
                stack.append(cur)
                cur = cur.left
            cur = stack.pop()             # node nhỏ nhất chưa thăm
            v = cur.val
            count = count + 1 if v == prev else 1
            if count > max_count:
                max_count, modes = count, [v]
            elif count == max_count:
                modes.append(v)
            prev = v
            cur = cur.right
        return modes

s = Solution()
print(s.findMode(build_tree([1, None, 2, 2])))   # [2]
print(s.findMode(build_tree([0])))               # [0]

[2]
[0]


## 10. Vì sao đúng?

1. **In-order của BST là dãy không giảm.** Quy nạp: in-order(node) = in-order(trái) + [node.val] + in-order(phải); mọi giá trị bên trái `≤ node.val ≤` mọi giá trị bên phải, và mỗi phần tự nó đã sắp xếp.
2. Trong dãy không giảm, các giá trị bằng nhau tạo thành **một đoạn liên tiếp duy nhất** → tần suất của `v` = độ dài đoạn của `v`.
3. **Invariant** sau mỗi bước: `count` = độ dài đoạn hiện tại tính tới phần tử vừa xét; `max_count` = độ dài lớn nhất trong các **tiền tố đoạn** đã thấy; `modes` = các giá trị đạt `max_count`. Khi một đoạn dài lên, nó đi qua mọi giá trị `1, 2, ..., L`, nên nó sẽ **bằng** rồi **vượt** kỷ lục đúng lúc. Một giá trị chỉ được `append` khi `count == max_count`, và mỗi đoạn chỉ chạm mức `max_count` hiện hành một lần → không bị thêm trùng.
4. Kết thúc: `max_count` là tần suất lớn nhất, `modes` chứa đúng các giá trị có đoạn dài bằng nó.

## 11. Độ phức tạp

`n` = số node, `h` = chiều cao cây (`log n ≤ h ≤ n`).

| Cách | Time | Space phụ | Ghi chú |
|---|---:|---:|---|
| Counter | `O(n)` | `O(n)` | không cần BST |
| In-order ra list + đếm đoạn | `O(n)` | `O(n)` | dễ hiểu |
| In-order đếm tại chỗ (đệ quy/stack) | `O(n)` | `O(h)` | đáp ứng follow up (không tính stack) |
| Morris | `O(n)` | `O(1)` | mỗi cạnh được đi qua tối đa vài lần |

(Danh sách kết quả `modes` không tính vào bộ nhớ phụ.)

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)   # đề cho phép thứ tự bất kỳ
    print(f"PASS {name}: {actual}")

s = Solution()
check("đề 1", s.findMode(build_tree([1, None, 2, 2])), [2])
check("đề 2", s.findMode(build_tree([0])), [0])
check("mọi giá trị khác nhau -> tất cả là mode", s.findMode(build_tree([2, 1, 3])), [1, 2, 3])
check("hai mode", s.findMode(build_tree([5, 3, 7, 3, 4, 6, 7])), [3, 7])
check("kỷ lục bị phá", s.findMode(build_tree([2, 1, 3, 1, 2, 3, 3])), [3])
check("toàn bộ giống nhau", s.findMode(build_tree([4, 4, 4, 4])), [4])
check("giá trị âm", s.findMode(build_tree([-1, -1, 0])), [-1])

PASS đề 1: [2]
PASS đề 2: [0]
PASS mọi giá trị khác nhau -> tất cả là mode: [1, 2, 3]
PASS hai mode: [3, 7]
PASS kỷ lục bị phá: [3]
PASS toàn bộ giống nhau: [4]
PASS giá trị âm: [-1]


In [16]:
import random

def bst_insert(root, v):
    # Chèn kiểu BST cho phép trùng (trùng đi sang phải). Dùng vòng lặp để tránh đệ quy sâu.
    node = TreeNode(v)
    if root is None:
        return node
    cur = root
    while True:
        if v < cur.val:
            if cur.left is None:
                cur.left = node; return root
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = node; return root
            cur = cur.right

random.seed(501)
for _ in range(300):
    root = None
    for v in [random.randint(-3, 3) for _ in range(random.randint(1, 25))]:
        root = bst_insert(root, v)
    expected = sorted(find_mode_counter(root))
    assert sorted(find_mode_inorder_list(root)) == expected
    assert sorted(find_mode_inorder_inplace(root)) == expected
    assert sorted(find_mode_morris(root)) == expected
    assert sorted(Solution().findMode(root)) == expected
print("300 BST ngẫu nhiên: 5 cách khớp nhau — PASS")

# Cây lệch 10^4 node (dạng đường thẳng) — bản stack không bị lỗi đệ quy
root = None
tail = None
for v in range(10000):
    node = TreeNode(v // 3)          # mỗi giá trị lặp 3 lần, trừ giá trị cuối
    if root is None:
        root = tail = node
    else:
        tail.right = node; tail = node
res = Solution().findMode(root)
print("cây lệch 10^4 node: số mode =", len(res), "| mode đầu:", res[:3])
assert len(res) == 3333 and res[:3] == [0, 1, 2]

300 BST ngẫu nhiên: 5 cách khớp nhau — PASS
cây lệch 10^4 node: số mode = 3333 | mode đầu: [0, 1, 2]


## 13. Lỗi người mới hay gặp

### Lỗi 1: quên reset `modes` khi có kỷ lục mới
```python
if count > max_count:
    max_count = count
    modes.append(v)      # SAI: các mode cũ (tần suất thấp hơn) vẫn còn
```

### Lỗi 2: chỉ cập nhật khi đoạn kết thúc nhưng quên đoạn cuối cùng
Nếu chỉ xử lý "khi `v != prev`" thì đoạn cuối cùng không bao giờ được xét.

### Lỗi 3: dùng pre-order cho cách đếm đoạn
Pre-order không cho dãy sắp xếp → các giá trị bằng nhau có thể bị tách rời.

### Lỗi 4: quên `nonlocal`
Gán `count = ...` trong hàm con tạo biến **cục bộ mới** → `UnboundLocalError`.

In [17]:
def buggy_no_reset(arr):
    prev, count, max_count, modes = None, 0, 0, []
    for v in arr:
        count = count + 1 if v == prev else 1
        if count > max_count:
            max_count = count
            modes.append(v)           # lỗi
        elif count == max_count:
            modes.append(v)
        prev = v
    return modes

def buggy_preorder_runs(root):
    arr = []
    preorder(root, arr)
    return modes_of_sorted(arr)

print("không reset    :", buggy_no_reset([1, 2, 2]), "| đúng:", modes_of_sorted([1, 2, 2]))
t = build_tree([2, 1, 3, 1, 2])          # in-order 1,1,2,2,3 ; pre-order 2,1,1,2,3
print("dùng pre-order :", buggy_preorder_runs(t), "| đúng:", sorted(find_mode_counter(t)))

def buggy_nonlocal():
    count = 0
    def inc():
        count += 1
    inc()
try:
    buggy_nonlocal()
except UnboundLocalError as e:
    print("quên nonlocal  :", type(e).__name__)

không reset    : [1, 2, 2] | đúng: [2]
dùng pre-order : [1] | đúng: [1, 2]
quên nonlocal  : UnboundLocalError


## 14. Mẹo nhận dạng pattern

- Thấy **BST** + câu hỏi về **thứ tự / giá trị liền kề / trùng lặp / hiệu nhỏ nhất / phần tử thứ k** → nghĩ ngay **in-order = dãy sắp xếp**.
- Thấy "**đếm tần suất trên dãy đã sắp xếp**" → không cần dict, chỉ cần `prev` + `count` (đếm theo đoạn — run-length).
- "Lấy **tất cả** phần tử đạt cực đại" → pattern `if > max: reset list; elif == max: append`.
- Follow up "không dùng thêm bộ nhớ" trên cây → in-order lặp hoặc **Morris traversal**.

## 15. Biến thể

### 15.1. Mode của một list đã sắp xếp (không có cây)
Đã làm ở `modes_of_sorted`.

### 15.2. Mode trên **cây nhị phân thường** (không phải BST)
In-order không còn sắp xếp → phải dùng `Counter` (Cách 1).

### 15.3. Kiểm tra một cây có là BST (cho phép trùng) không
In-order phải **không giảm**.

In [18]:
def is_bst_allow_dup(root):
    arr = []
    inorder(root, arr)
    return all(arr[i] <= arr[i + 1] for i in range(len(arr) - 1))

print(is_bst_allow_dup(build_tree([5, 3, 7, 3, 4, 6, 7])))   # True
print(is_bst_allow_dup(build_tree([5, 6, 7])))               # False

True
False


## 16. Bài tập (đáp án ở cuối)

### Bài 1 — Tính tay
Cho BST `[6, 2, 8, 2, 4, 8, 9, None, None, 4, 4]`. Viết dãy in-order, lập bảng `count/max_count/modes` như mục 7 và cho biết các mode.

### Bài 2 — Điền TODO cho hàm `visit`

In [19]:
def exercise_modes_of_sorted(arr):
    prev, count, max_count, modes = None, 0, 0, []
    for v in arr:
        # TODO 1: cập nhật count
        # TODO 2: kỷ lục mới -> reset modes
        # TODO 3: hoà kỷ lục -> thêm v
        # TODO 4: cập nhật prev
        pass
    return modes

# assert exercise_modes_of_sorted([1, 1, 2, 3, 3]) == [1, 3]

### Bài 3 — Tần suất lớn nhất
Viết `max_frequency_bst(root)` trả về **số lần** xuất hiện của mode (không trả giá trị).

In [20]:
def max_frequency_bst(root):
    # TODO
    pass

# assert max_frequency_bst(build_tree([1, None, 2, 2])) == 2

### Bài 4 — Số giá trị phân biệt trong BST
Dùng in-order + `prev`, **không dùng set**.

In [21]:
def count_distinct_bst(root):
    # TODO
    pass

# assert count_distinct_bst(build_tree([5, 3, 7, 3, 4, 6, 7])) == 5

### Bài 5 — Mode trên cây nhị phân thường
Cây không có tính chất BST. Trả về các mode (thứ tự bất kỳ).

In [22]:
def find_mode_any_tree(root):
    # TODO
    pass

# assert sorted(find_mode_any_tree(build_tree([1, 2, 1, 2, 3]))) == [1, 2]

### Bài 6 — Mini challenge: các đoạn dài nhất
Cho list đã sắp xếp, trả về list các cặp `(giá_trị, tần_suất)` của **k** giá trị xuất hiện nhiều nhất (hoà thì giá trị nhỏ trước), chỉ duyệt đếm đoạn một lần rồi sắp xếp.

In [23]:
def top_k_runs(arr, k):
    # TODO
    pass

# assert top_k_runs([1, 1, 2, 3, 3, 3, 4, 4], 2) == [(3, 3), (1, 2)]

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Dựng cây bằng tay theo tầng | In-order: trái → node → phải | Kiểm tra lại bằng `trace_find_mode(build_tree(...))` |
| 2 | Xem mục 4 | `count = count + 1 if v == prev else 1` | `if count > max_count: ... modes = [v]` / `elif count == max_count: modes.append(v)` |
| 3 | Giống lời giải chính nhưng bỏ `modes` | chỉ giữ `max_count` | trả `max_count` |
| 4 | Giá trị mới xuất hiện khi `v != prev` | đếm số lần đổi giá trị | lần đầu `prev = None` cũng tính là giá trị mới |
| 5 | In-order không còn sắp xếp | dùng `Counter` | `best = max(freq.values())` |
| 6 | Đếm đoạn → list `(v, len)` | `sorted(runs, key=lambda p: (-p[1], p[0]))` | lấy `[:k]` |

## 18. Đáp án tham khảo

**Bài 1:** cây `[6, 2, 8, 2, 4, 8, 9, None, None, 4, 4]` có in-order `2, 2, 4, 4, 4, 6, 8, 8, 9` → mode là `[4]` (3 lần). Kiểm tra:

In [24]:
t1 = build_tree([6, 2, 8, 2, 4, 8, 9, None, None, 4, 4])
show_tree(t1)
seq = []; inorder(t1, seq); print("in-order:", seq)
trace_find_mode(t1)

        9
    8
        8
6
            4
        4
            4
    2
        2
in-order: [2, 2, 4, 4, 4, 6, 8, 8, 9]
bước 1: v=  2 prev=None count=1 max=1 -> kỷ lục mới modes=[2]
bước 2: v=  2 prev=   2 count=2 max=2 -> kỷ lục mới modes=[2]
bước 3: v=  4 prev=   2 count=1 max=2 -> thấp hơn   modes=[2]
bước 4: v=  4 prev=   4 count=2 max=2 -> hoà kỷ lục modes=[2, 4]
bước 5: v=  4 prev=   4 count=3 max=3 -> kỷ lục mới modes=[4]
bước 6: v=  6 prev=   4 count=1 max=3 -> thấp hơn   modes=[4]
bước 7: v=  8 prev=   6 count=1 max=3 -> thấp hơn   modes=[4]
bước 8: v=  8 prev=   8 count=2 max=3 -> thấp hơn   modes=[4]
bước 9: v=  9 prev=   8 count=1 max=3 -> thấp hơn   modes=[4]


[4]

In [25]:
def answer_exercise_modes_of_sorted(arr):
    prev, count, max_count, modes = None, 0, 0, []
    for v in arr:
        count = count + 1 if v == prev else 1
        if count > max_count:
            max_count, modes = count, [v]
        elif count == max_count:
            modes.append(v)
        prev = v
    return modes

def answer_max_frequency_bst(root):
    arr = []; inorder(root, arr)
    prev, count, best = None, 0, 0
    for v in arr:
        count = count + 1 if v == prev else 1
        best = max(best, count)
        prev = v
    return best

def answer_count_distinct_bst(root):
    arr = []; inorder(root, arr)
    prev, distinct = None, 0
    for i, v in enumerate(arr):
        if i == 0 or v != prev:
            distinct += 1
        prev = v
    return distinct

def answer_find_mode_any_tree(root):
    vals = []; preorder(root, vals)
    freq = Counter(vals)
    best = max(freq.values())
    return [v for v, f in freq.items() if f == best]

def answer_top_k_runs(arr, k):
    runs = []
    for v in arr:
        if runs and runs[-1][0] == v:
            runs[-1] = (v, runs[-1][1] + 1)
        else:
            runs.append((v, 1))
    return sorted(runs, key=lambda p: (-p[1], p[0]))[:k]

assert answer_exercise_modes_of_sorted([1, 1, 2, 3, 3]) == [1, 3]
assert answer_max_frequency_bst(build_tree([1, None, 2, 2])) == 2
assert answer_count_distinct_bst(build_tree([5, 3, 7, 3, 4, 6, 7])) == 5
assert sorted(answer_find_mode_any_tree(build_tree([1, 2, 1, 2, 3]))) == [1, 2]
assert answer_top_k_runs([1, 1, 2, 3, 3, 3, 4, 4], 2) == [(3, 3), (1, 2)]
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
prev, count, max_count, modes = None, 0, 0, []
for v in inorder_values:                 # BST -> in-order = dãy không giảm
    count = count + 1 if v == prev else 1
    if count > max_count:                # kỷ lục mới
        max_count, modes = count, [v]
    elif count == max_count:             # hoà kỷ lục
        modes.append(v)
    prev = v
```

Câu nhớ: **"BST → in-order → dãy sắp xếp → đếm đoạn bằng nhau."**

### Checklist
- [ ] Duyệt **in-order** (trái → node → phải), không phải pre/post-order.
- [ ] Kỷ lục mới thì **reset** `modes = [v]`; hoà thì `append`.
- [ ] Cập nhật `prev = v` sau mỗi node.
- [ ] Hàm lồng nhau gán biến ngoài → `nonlocal`.
- [ ] Cây có thể sâu tới `10^4` → ưu tiên in-order bằng stack (hoặc tăng `sys.setrecursionlimit`).
- [ ] Time `O(n)`, Space phụ `O(h)` (Morris: `O(1)`).